# AgentCore Runtime의 Runtime Context 및 Session Management 이해

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime의 runtime context와 session management를 이해하고 활용하는 방법을 알아봅니다. 이 예제에서는 AgentCore Runtime이 session을 처리하고 여러 호출에서 context를 유지하는 방식과 에이전트가 context object를 통해 Runtime 정보에 액세스하는 방법을 보여줍니다.

Amazon Bedrock AgentCore Runtime은 각 사용자 상호 작용에 격리된 session을 제공합니다. 이를 통해 서로 다른 사용자 간의 완전한 보안 격리를 보장하면서 에이전트가 여러 호출에서 context와 state를 유지할 수 있습니다.

### 튜토리얼 세부 정보

|항목| 세부 정보|
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | Context 및 Session Management|
| Agent 유형          | Single         |
| Agentic Framework   | Strands Agents |
| LLM 모델            | Anthropic Claude Haiku 4.5 |
| 튜토리얼 구성 요소  | Runtime Context, Session Management, AgentCore Runtime, Strands Agent 및 Amazon Bedrock 모델 |
| 튜토리얼 분야       | 산업 공통                                                                        |
| 예제 난이도         | 중급                                                                              |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3|

### 튜토리얼 아키텍처

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime이 session을 관리하고 에이전트에 context를 제공하는 방식을 살펴봅니다. 다음 내용을 보여줍니다.

1. **Session Continuity**: 동일한 session ID가 여러 호출에서 context를 유지하는 방식
2. **Context Object**: 에이전트가 context parameter를 통해 Runtime 정보에 액세스하는 방식
3. **Session Isolation**: 서로 다른 session ID가 완전히 격리된 환경을 생성하는 방식
4. **Payload Flexibility**: payload를 통해 에이전트에 custom 데이터를 전달하는 방식

시연을 위해 이러한 session management 기능을 보여주는 Strands Agent를 사용합니다.

    
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>

### 튜토리얼 주요 기능

* **Session 기반 Context Management**: AgentCore Runtime이 session 내에서 context를 유지하는 방식 이해
* **Runtime Session Lifecycle**: session 생성, 유지, 종료 학습
* **Context Object Access**: context parameter를 통해 session ID 같은 Runtime 정보에 액세스
* **Session Isolation**: 서로 다른 session이 완전한 격리를 제공하는 방식 시연
* **Payload Handling**: custom payload 구조를 통한 유연한 데이터 전달
* **Cross-invocation State**: 동일한 session 내 여러 호출에서 agent state 유지

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* 실행 중인 Docker

## Amazon Bedrock AgentCore Runtime Session 이해

코드를 살펴보기 전에 Amazon Bedrock AgentCore Runtime이 session을 관리하는 방식을 이해해야 합니다.

### Session Isolation 및 보안

AgentCore Runtime은 전용 microVM을 통해 **완전한 session isolation**을 제공합니다.

- **전용 리소스**: 각 session은 CPU, memory, filesystem이 격리된 자체 microVM에서 실행
- **보안 경계**: 사용자 session 간 완전한 분리로 데이터 오염 방지
- **결정론적 정리**: session 완료 후 microVM이 종료되고 memory가 정리됨

### Session Lifecycle

AgentCore Runtime의 session은 다음 lifecycle을 따릅니다.

1. **생성**: 첫 호출 시 고유한 `runtimeSessionId`로 새 session 생성
2. **Active State**: session이 request를 처리하고 context 유지
3. **Idle State**: session이 context를 보존하면서 다음 호출을 대기
4. **종료**: 다음 이유로 session 종료
   - 비활성 상태(15 minutes)
   - 최대 수명(8 hours)
   - Health check 실패

**팁: Session Lifecycle은 [구성할 수 있으므로](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-lifecycle-settings.html#configuration-attributes) 비즈니스 요구 사항에 맞게 구성해야 합니다.**

### Context Persistence

AgentCore Runtime은 session 내에서 다음 항목을 유지합니다.
- **Conversation History**: 이전 상호 작용과 응답
- **Application State**: 실행 중 생성된 variable과 object
- **File System**: session 중 생성되거나 수정된 모든 파일
- **Environment Variables**: custom 설정 및 구성

### Session Management 모범 사례

- **고유한 Session ID**: 각 사용자 또는 대화에 고유한 session ID 생성
- **Context 재사용**: 관련 호출에 동일한 session ID를 사용하여 context 유지
- **Session 경계**: 서로 다른 사용자 또는 관련 없는 대화에 서로 다른 session ID 사용
- **일시적 특성**: 영구 데이터 저장에 session을 사용하지 않음(persistence에는 AgentCore Memory 사용)

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## AgentCore Runtime 배포를 위한 에이전트 준비

session management와 context handling을 보여주기 위해 에이전트를 AgentCore Runtime에 배포합니다. 에이전트는 다음 방법을 보여줍니다.

1. **Runtime Context 액세스**: `context` parameter를 사용하여 session 정보 가져오기
2. **Custom Payload 처리**: payload를 통해 전달된 structured data 처리
3. **Session State 유지**: session 내 사용자 상호 작용 추적
4. **Session 경계 시연**: 서로 다른 session이 격리되는 방식 확인

### Context Object 이해

AgentCore Runtime의 `context` object는 현재 실행 환경에 관한 유용한 정보를 제공합니다.

- **session_id**: 현재 runtime session identifier
- **Runtime Metadata**: Runtime 환경 정보
- **Execution Details**: 현재 호출에 관한 context

### Context Handling을 사용하는 Strands Agent

session management와 context handling을 보여주는 구현을 살펴봅니다.

In [ ]:
%%writefile strands_claude_context.py
from strands import Agent, tool
from strands_tools import calculator # calculator tool 가져오기
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel
import asyncio
from datetime import datetime

app = BedrockAgentCoreApp()

# custom tool 생성 
@tool
def weather():
    """ Get weather """ # dummy 구현
    return "sunny"

@tool
def get_time():
    """ Get current time """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[
        calculator, weather, get_time
    ],
    system_prompt="""
    You're a helpful assistant. You can do simple math calculations, 
    tell the weather, and provide the current time.
    Always start by acknowledging the user's name 
    """
)

def get_user_name(user_id):
    users = {
        "1": "Maira",
        "2": "Mani",
        "3": "Mark",
        "4": "Ishan",
        "5": "Dhawal"
    }
    return users[user_id]
    
@app.entrypoint
def strands_agent_bedrock_handling_context(payload, context):
    """
    컨텍스트 처리와 세션 관리를 보여 주는 AgentCore Runtime 엔트리포인트입니다.
    
    매개변수:
        payload: 사용자 데이터와 요청 정보가 포함된 입력 payload
        context: 세션 및 실행 정보가 포함된 runtime context 객체
    
    반환값:
        str: 컨텍스트 정보가 반영된 에이전트 응답
    """
    user_input = payload.get("prompt")
    user_id = payload.get("user_id")
    user_name = get_user_name(user_id)
    
    # runtime context 정보에 액세스
    print("=== Runtime Context Information ===")
    print("User id:", user_id)
    print("User Name:", user_name)
    print("User input:", user_input)
    print("Runtime Session ID:", context.session_id)
    print("Context Object Type:", type(context))
    print("=== End Context Information ===")
    
    # context 정보가 포함된 personalized prompt 생성
    prompt = f"""My name is {user_name}. Here is my request: {user_input}
    
    Additional context: This is session {context.session_id}. 
    Please acknowledge my name and provide assistance."""
    
    response = agent(prompt)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## AgentCore Runtime의 Session Management 이해

위 코드는 AgentCore Runtime이 session을 관리하고 에이전트에 context를 제공하는 방식에 관한 여러 핵심 개념을 보여줍니다.

### Context Object 구조

entrypoint 함수의 `context` parameter를 통해 Runtime 정보에 액세스할 수 있습니다.

```python
@app.entrypoint
def strands_agent_bedrock_handling_context(payload, context):
    # session 정보에 액세스
    session_id = context.session_id
    # agent logic에서 context 정보 사용
```

### Session 연속성의 이점

AgentCore Runtime은 단일 session 내에서 다음을 제공합니다.

1. **Persistent Environment**: 여러 호출에서 variable과 state 유지
2. **Context Preservation**: 에이전트가 이전 상호 작용 참조 가능
3. **리소스 재사용**: 초기화된 모델과 tool을 load 상태로 유지
4. **성능 이점**: 후속 호출의 cold start 시간 단축

### Session 격리 보장

AgentCore Runtime은 session 간 완전한 격리를 보장합니다.

- **보안**: 각 session이 리소스가 격리된 자체 microVM에서 실행
- **개인정보 보호**: 서로 다른 사용자 session 간 데이터 유출 방지
- **안정성**: 한 session의 문제가 다른 session에 영향을 주지 않음
- **정리**: session 종료 후 memory를 완전히 정리

### Payload Flexibility

`payload` parameter를 사용하면 데이터를 유연하게 전달할 수 있습니다.

```python
# payload 구조 예제
payload = {
    "prompt": "User's question",
    "user_id": "1",
    "preferences": {...},
    "context_data": {...}
}
```

이를 통해 Runtime에서 제공하는 session context를 유지하면서 client와 에이전트 간에 풍부한 structured communication이 가능합니다.

### AgentCore Runtime 배포 구성

다음으로 starter toolkit을 사용하여 entrypoint, 방금 생성한 execution role, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 또한 시작할 때 Amazon ECR repository를 자동으로 생성하도록 starter toolkit을 구성합니다.

configure 단계에서 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
region

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude_context.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_claude_context",
)

### Context-aware Agent를 AgentCore Runtime에 시작

Dockerfile이 준비되었으므로 context-aware agent를 AgentCore Runtime에 시작합니다. 이 과정에서 Amazon ECR repository와 AgentCore Runtime이 생성됩니다.

에이전트는 AgentCore Runtime이 session을 관리하고 에이전트에 context 정보를 제공하는 방식을 보여줍니다.

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()

### AgentCore Runtime 상태 확인
AgentCore Runtime을 배포했으므로 배포 상태를 확인합니다.

In [ ]:
status_response = agentcore_runtime.status()
status_response.endpoint["status"]

## Session Management 및 Context Handling 시연

서로 다른 scenario를 테스트하여 AgentCore Runtime의 주요 session management 기능을 시연합니다.

### Scenario 1: Session 연속성
여러 호출에 동일한 session ID를 사용하여 context가 유지되는 방식을 보여줍니다.

### Scenario 2: Session 격리
서로 다른 session ID를 사용하여 session 간 완전한 격리를 보여줍니다.

### Scenario 3: Context 정보 액세스
에이전트가 runtime context 정보에 액세스하는 방법을 보여줍니다.

<div style="text-align:left">
    <img src="images/invoke.png" width="85%"/>
</div>

이제 ID = 1인 사용자의 첫 번째 session을 생성합니다.

In [ ]:
import uuid
from IPython.display import Markdown, display

# session continuity 시연용 session ID 생성
session_id = uuid.uuid4()
print(f"📋 Starting Session 1: {session_id}")
print("👤 User: Maira (ID: 1)")
print("❓ First question about weather\n")

invoke_response = agentcore_runtime.invoke(
    {"prompt": "How is the weather outside?", "user_id": "1"},
    session_id=str(session_id),
)

response_data = invoke_response["response"][0]
display(Markdown(response_data))

동일한 session에서 계속 질문할 수 있습니다.

In [ ]:
# session continuity를 보여주도록 동일한 session ID로 계속
print(f"🔄 Continuing Session 1: {session_id}")
print("👤 Same user: Maira (ID: 1)")
print("❓ Follow-up question about math\n")

invoke_response = agentcore_runtime.invoke({"prompt": "How much is 2X5?", "user_id": "1"}, session_id=str(session_id))

response_data = invoke_response["response"][0]
display(Markdown(response_data))

에이전트가 동일한 session에서 계속 작동하므로 이전 상호 작용 정보를 유지합니다.

In [ ]:
# 동일한 session ID로 계속 - 에이전트가 이전 계산을 기억하는지 확인
print(f"🔄 Continuing Session 1: {session_id}")
print("👤 Same user: Maira (ID: 1)")
print("❓ Building on previous answer - demonstrates context continuity\n")

invoke_response = agentcore_runtime.invoke({"prompt": "and that plus 34?", "user_id": "1"}, session_id=str(session_id))

response_data = invoke_response["response"][0]
display(Markdown(response_data))

#### Session 1 중지

이 상호 작용이 끝났으므로 `stop_runtime_session` 명령으로 session을 중지할 수 있습니다. 

[StopRuntimeSession](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-stop-session.html) operation을 사용하면 적절한 리소스 정리와 session lifecycle 관리를 위해 활성 AgentCore Runtime session을 즉시 종료할 수 있습니다.

이 명령은 사용자가 상호 작용하던 활성 session(microVM)을 종료하므로 사용자가 새 대화를 시작하면 이전 context가 사라집니다. 비즈니스 요구 사항에 맞게 구성할 수 있으며 현재 workload가 끝난 후 사용하는 것이 좋습니다.

In [ ]:
# --- Inline Session Lifecycle 시연(Scenario 1) ---
# session continuity 시연을 마쳤으므로 이 session 중지
# stop_runtime_session은 이 특정 session의 microVM 리소스를 해제함
# 이때 Runtime은 새 session에 사용할 수 있도록 계속 실행됨

import boto3

agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

agentcore_client.stop_runtime_session(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=str(session_id),
    qualifier="DEFAULT",
)
print(f"✅ Session 1 '{session_id}' stopped — microVM resources released")

사용자 1의 session이 종료되었으므로 새로운 상호 작용은 이전 기록이 없는 새 session과 microVM을 생성합니다.

In [ ]:
# 새 SESSION - session isolation 시연
# context가 사라짐을 보여주도록 완전히 새로운 session ID 생성
new_session_id = uuid.uuid4()
print(f"🆕 Starting NEW Session 2: {new_session_id}")
print("👤 Same user: Maira (ID: 1)")
print("❓ Attempting to reference previous calculation - should fail due to session isolation\n")

invoke_response = agentcore_runtime.invoke({"prompt": "And plus 10?", "user_id": "1"}, session_id=str(new_session_id))

response_data = invoke_response["response"][0]
display(Markdown(response_data))

마지막으로 이 새 session을 종료합니다.

session isolation을 보여주기 위해 사용자 1의 두 번째 session을 종료하기 전에 사용자 2와 새 대화를 시작합니다.

In [ ]:
# 새 SESSION 및 USER - 완전한 격리 시연
different_user_session = uuid.uuid4()
print(f"🆕 Starting Session 3: {different_user_session}")
print("👤 Different user: Mani (ID: 2)")
print("❓ Same question as first user - demonstrates user isolation\n")

invoke_response = agentcore_runtime.invoke(
    {"prompt": "How is the weather?", "user_id": "2"},
    session_id=str(different_user_session),
)

response_data = invoke_response["response"][0]
display(Markdown(response_data))

### 두 Session 중지

개별 session이 더 이상 필요하지 않으면 중지해야 합니다.

이렇게 하면 해당 session의 microVM 리소스가 해제됩니다. 

In [ ]:
# --- Inline Session Lifecycle 시연(Scenario 2) ---
# isolation 시연 session 중지. 설계상 context는 사라졌지만
# microVM은 계속 실행 중이며, 중지하면 해당 리소스가 해제됨

agentcore_client.stop_runtime_session(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=str(new_session_id),
    qualifier="DEFAULT",
)
print(f"✅ Session 2 '{new_session_id}' stopped — microVM resources released")

In [ ]:
# --- Inline Session Lifecycle 시연(Scenario 3) ---
# lifecycle 시연을 완료하도록 다른 사용자 session 중지

agentcore_client.stop_runtime_session(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=str(different_user_session),
    qualifier="DEFAULT",
)
print(f"✅ Session 3 '{different_user_session}' stopped — microVM resources released")
print()
print("All demo sessions stopped. The runtime is still alive for new sessions.")

### Lifecycle Configuration 시연 — 더 짧은 Idle Timeout

더 짧은 idle timeout이 적용된 두 번째 Runtime을 생성하여
lifecycle configuration의 작동 방식을 보여줍니다. 기존 Runtime과 더 짧은
timeout이 적용된 새 Runtime이 함께 존재합니다.

더 짧은 idle timeout은 더 이상 활성 상태가 아닌 session을 자동으로 종료하여
원치 않는 비용을 방지하는 데 도움이 됩니다. production에서는 workload에
적합한 timeout을 선택해야 합니다.

In [ ]:
# --- 더 짧은 idle timeout이 적용된 두 번째 Runtime 생성 ---
# 첫 번째 Runtime과 새 Runtime이 함께 존재함
# 이를 통해 동일한 계정에서 서로 다른 사용 사례에 맞게
# 서로 다른 timeout 값을 구성하는 방법을 보여줌

agentcore_runtime_short_timeout = Runtime()

response = agentcore_runtime_short_timeout.configure(
    entrypoint="strands_claude_context.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_claude_context_short_timeout",
)
print("✅ Configured second runtime")

In [ ]:
# --- 두 번째 Runtime 시작 ---
launch_result_short = agentcore_runtime_short_timeout.launch()
print("✅ Second runtime launched")
print(f"   Agent ID: {launch_result_short.agent_id}")
print(f"   Agent ARN: {launch_result_short.agent_arn}")

In [ ]:
# --- boto3 update_agent_runtime을 통해 더 짧은 idle timeout 설정 ---
# Starter Toolkit은 lifecycleConfiguration을 직접 노출하지 않으므로
# 생성 후 boto3를 사용하여 idleRuntimeSessionTimeout 설정
# 기본값은 900s(15 min)이며 이 시연에서는 300s(5 min)로 설정
# update_agent_runtime에는 전체 config가 필요하므로 먼저 읽음

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

# 필수 field를 다시 전달하도록 현재 runtime config 가져오기
runtime_info = agentcore_control_client.get_agent_runtime(agentRuntimeId=launch_result_short.agent_id)

agentcore_control_client.update_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id,
    agentRuntimeArtifact=runtime_info["agentRuntimeArtifact"],
    roleArn=runtime_info["roleArn"],
    networkConfiguration=runtime_info["networkConfiguration"],
    lifecycleConfiguration={
        "idleRuntimeSessionTimeout": 300  # 5 minutes — 시연용 더 짧은 timeout
    },
)
print("✅ Updated idle timeout to 300s (5 minutes) via lifecycleConfiguration")

In [ ]:
# 준비 상태인지 확인
status_response = agentcore_runtime_short_timeout.status()
status = status_response.endpoint["status"]
status

In [ ]:
# --- 작동 여부를 확인하도록 두 번째 Runtime 호출 ---
# 이 session은 5 minutes 동안 비활성 상태이면 자동 종료되며
# 기본 timeout 15 minutes를 사용하는 기존 Runtime과 다름

short_timeout_session = uuid.uuid4()
print(f"📋 Invoking runtime with shorter idle timeout (session: {short_timeout_session})")
print("   This session will auto-terminate after 5 minutes of inactivity\n")

invoke_response = agentcore_runtime_short_timeout.invoke(
    {"prompt": "What time is it?", "user_id": "1"},
    session_id=str(short_timeout_session),
)

response_data = invoke_response["response"][0]
display(Markdown(response_data))

print("\n✅ Both runtimes are running simultaneously:")
print(f"   Original runtime: {launch_result.agent_id} (default timeout)")
print(f"   Short-timeout runtime: {launch_result_short.agent_id} (5 min idle timeout)")
print("   The short-timeout session will auto-stop after 5 minutes of inactivity,")
print("   releasing microVM resources without manual intervention.")

## Session Management 결과 이해

위 시연에서는 AgentCore Runtime session management의 여러 핵심 측면을 보여줍니다.

### 1. Session 연속성(Session 1)
- **첫 번째 호출**: 에이전트가 날씨 질문에 응답하고 사용자 이름을 인식
- **두 번째 호출**: 에이전트가 계산 수행(2×5=10)
- **세 번째 호출**: 에이전트가 이전 결과 참조("that plus 34" = 44)

**핵심 학습 내용**: 에이전트는 동일한 session 내 여러 호출에서 context를 유지하여 이전 상호 작용의 계산 결과를 기억했습니다.

### 2. Session 격리(Session 2)
- **새 session ID**: 완전히 새로운 session 생성
- **동일한 사용자**: 동일한 user ID를 사용하지만 다른 session 사용
- **Context 손실**: 에이전트가 이전 계산을 참조할 수 없음

**핵심 학습 내용**: 동일한 사용자라도 새 session은 이전 context에 액세스할 수 없는 완전히 격리된 환경을 생성합니다.

### 3. 사용자 및 Session 격리(Session 3)
- **다른 사용자**: Maira가 아닌 Mani
- **새 session**: 이전 session과 완전히 격리
- **새 context**: 에이전트가 초기 state로 시작

**핵심 학습 내용**: 각 session은 완전한 격리를 제공하여 서로 다른 사용자와 상호 작용 간의 개인정보 보호 및 보안을 보장합니다.

### 4. Context Object 사용
모든 호출에서 에이전트는 다음 작업을 수행했습니다.
- `context.session_id`를 통해 runtime context에 액세스
- custom payload 데이터 처리(`user_id`, `prompt`)
- logging 및 debugging 정보 유지

**핵심 학습 내용**: context object는 에이전트가 기능 향상과 debugging에 사용할 수 있는 유용한 Runtime 정보를 제공합니다.

### 시연한 Session Management 모범 사례

1. 대화 연속성을 위해 **일관된 session ID 사용**
2. 서로 다른 사용자 또는 대화에 **고유한 session ID 생성**
3. 향상된 agent 동작을 위해 **context 정보 활용**
4. session 간 persistence를 가정하지 않고 **session 경계를 고려하여 설계**
5. session이 변경되거나 만료될 때 **context 손실을 원활하게 처리**

## Session Lifecycle 모범 사례

AgentCore Runtime 비용은 vCPU와 Memory를 기준으로 책정됩니다. 원치 않는 비용을 방지하려면 session을 명시적으로 중지하거나 적절한 idle timeout을 구성하여 session이 종료되도록 하는 것이 좋습니다.

비용을 효과적으로 관리하려면 다음을 수행합니다.

- **idle timeout 구성**: session을 생성할 때 적절한 idle timeout을 설정하여 비활성 session을 자동으로 중지합니다. 사용 사례에 맞는 값(예: 개발/테스트에는 짧게, production workload에는 길게)을 선택합니다.
- **완료 후 session 중지**: Runtime은 새 session에 사용할 수 있도록 유지하면서 `stop_runtime_session`으로 특정 session의 microVM 리소스를 해제합니다.

## 리소스 정리

이제 AgentCore Runtime과 관련 리소스를 정리합니다. 원치 않는 비용을 방지하도록 Runtime을 먼저 삭제한 다음 ECR repository 같은 지원 리소스를 정리합니다.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
# --- microVM 리소스를 해제하도록 활성 session 중지 ---
import boto3

agentcore_client = boto3.client("bedrock-agentcore", region_name=region)
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

# 기존 Runtime의 session 중지(session continuity + isolation 시연)
for sid in [session_id, new_session_id, different_user_session]:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=launch_result.agent_arn,
            runtimeSessionId=str(sid),
            qualifier="DEFAULT",
        )
        print(f"✅ Session '{sid}' stopped")
    except Exception as e:
        print(f"⚠️ Failed to stop session '{sid}': {e}")

# 짧은 timeout Runtime 시연의 session 중지
try:
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=launch_result_short.agent_arn,
        runtimeSessionId=str(short_timeout_session),
        qualifier="DEFAULT",
    )
    print(f"✅ Short-timeout session '{short_timeout_session}' stopped")
except Exception as e:
    print(f"⚠️ Failed to stop short-timeout session: {e}")

# --- 두 Runtime 모두 삭제 ---
# 기존 Runtime
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Original runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete original runtime: {e}")

# 짧은 timeout Runtime
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result_short.agent_id,
    )
    print(f"✅ Short-timeout runtime '{launch_result_short.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete short-timeout runtime: {e}")

# --- ECR repository 삭제 ---
for ecr_uri in set([launch_result.ecr_uri, launch_result_short.ecr_uri]):
    try:
        ecr_client.delete_repository(repositoryName=ecr_uri.split("/")[1], force=True)
        print(f"✅ ECR repository '{ecr_uri.split('/')[1]}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete ECR repository: {e}")

# 축하합니다!

Amazon Bedrock AgentCore Runtime으로 session management와 context handling을 성공적으로 구현하고 테스트했습니다. 

## 학습 내용

### Session Management 기본 사항
* **Session Continuity**: 동일한 session ID가 여러 호출에서 context를 유지하는 방식
* **Session Isolation**: 서로 다른 session ID가 완전히 격리된 환경을 생성하는 방식
* **Context Preservation**: 에이전트가 state를 유지하고 이전 상호 작용을 참조하는 방식
* **보안 경계**: AgentCore Runtime이 사용자 간 완전한 격리를 보장하는 방식

### Runtime Context 처리
* **Context Object Access**: `context` parameter를 통해 Runtime 정보에 액세스하는 방법
* **Session 정보**: agent logic에서 session ID를 가져와 사용하는 방법
* **Payload 처리**: custom payload를 통해 전달된 structured data를 처리하는 방법
* **Runtime Metadata**: 에이전트가 실행 환경 정보에 액세스하는 방법

### AgentCore Runtime 아키텍처
* **MicroVM Isolation**: 각 session이 격리된 자체 microVM에서 실행
* **리소스 관리**: session별 전용 CPU, memory, filesystem
* **보안 모델**: session 종료 후 memory를 완전히 정리
* **Lifecycle Management**: session state(active, idle, terminated) 및 timeout

### 모범 사례 구현
* **Session ID 생성**: 서로 다른 대화에 고유한 identifier 생성
* **Context 활용**: 향상된 agent 동작을 위해 runtime context 활용
* **State Management**: ephemeral state와 persistent state 이해
* **오류 처리**: context 손실과 session 경계를 원활하게 처리